# Notebook 1 — Full Training + Stratified 3:7 Cross-Class Forget Split

**Paper:** *An Illusion of Unlearning? Assessing Machine Unlearning Through Internal Representations*  
(Gao, Unal, Rangamani, Zhu — AISTATS 2026 · arXiv 2604.08271)

**Evaluation protocol — 3 metrics (matches paper's Table 1/3/6):**

| Metric | Description |
|--------|-------------|
| **Output** | Standard forward pass through full model (encoder + classifier head) |
| **Linear Probe** | Freeze encoder → train fresh linear classifier on full train set → report retain/forget accuracy |
| **NCC** | Freeze encoder → compute per-class mean features → nearest-class-center classify |

All three metrics are computed over **sample index sets** (not class labels),  
because the 3:7 forget split is cross-class: forget samples come from every class.

**Outputs:**

| Stage | Output | Path |
|-------|--------|------|
| **C** | Θ_o trained on full training set | `checkpoints/cmf_benchmark/pre_train/<tag>.pt` |
| **D** | 3-seed stratified 3:7 split files | `checkpoints/cmf_benchmark/splits/forget_indices_seed{s}.json` |
| **E** | `cmf_benchmark_config.json` | `checkpoints/cmf_benchmark/cmf_benchmark_config.json` |

> After this notebook finishes: publish `checkpoints/cmf_benchmark/` as a Kaggle dataset, then attach to NB2/3/4.

> Set `TEST_MODE=True` for a ~1-min CPU check. Full mode: 300-epoch pretrain on T4 ≈ 2-3 h.

## A. Environment Setup

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, random, argparse, collections, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

## B. Configuration

| Mode | `TEST_MODE` | Data | Epochs | Time |
|------|------------|------|--------|------|
| **Test** | `True` | 1% | 1 | ~1 min CPU |
| **Full** | `False` | 100% | 300 | ~2-3 h T4 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FLIP to False for the real experiment run
TEST_MODE     = True
TEST_FRACTION = 0.01
# ══════════════════════════════════════════════════════════════════════

PREV_RUN_DATASET_DIR = None  # e.g. '/kaggle/input/cmf_benchmark-notebook1' to skip retraining

DATASET   = 'cifar10'
ARCH      = {'cifar10': 'resnet18', 'cifar100': 'resnet18', 'tinyimagenet': 'resnet50'}[DATASET]
IS_VIT    = False
DATA_PATH = '/kaggle/working/data'

SPLIT_SEEDS     = [0, 1, 2]
FORGET_FRACTION = 0.30  # exactly 30% of EACH class -> forget set

_TOTAL     = {'cifar10': 50000, 'cifar100': 50000, 'tinyimagenet': 100000}
_PER_CLASS = {'cifar10': 5000,  'cifar100': 500,   'tinyimagenet': 500}
NUM_CLASSES_MAP = {'cifar10': 10, 'cifar100': 100, 'tinyimagenet': 200}

if TEST_MODE:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 1, 8, 1
else:
    PRETRAIN_LR, PRETRAIN_EPOCHS, PRETRAIN_BS, PRETRAIN_PATIENCE = 0.05, 300, 128, 50

CKPT_ROOT = '/kaggle/working/checkpoints/cmf_benchmark'
SPLIT_DIR = f'{CKPT_ROOT}/splits'
os.makedirs(CKPT_ROOT, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

_MODE_TAG = 'test' if TEST_MODE else 'full'
print(f'Mode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Pretrain epochs={PRETRAIN_EPOCHS}  Forget fraction={FORGET_FRACTION}')

## C-helpers. Args & Data

In [ ]:
from utils import get_dataset, get_model, test, SubSet
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw): return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=None, class_label_names=None,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5, min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='cmf_benchmark',
    )
    d.update(ov)
    return argparse.Namespace(**d)

def _find_ckpt(rel):
    local = f'{CKPT_ROOT}/{rel}'
    if os.path.exists(local): return local
    if PREV_RUN_DATASET_DIR:
        prev = f'{PREV_RUN_DATASET_DIR}/{rel}'
        if os.path.exists(prev): return prev
    return None

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
NUM_CLASSES       = args_base.num_classes
CLASS_LABEL_NAMES = args_base.class_label_names
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, frac, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_cls = collections.defaultdict(list)
        for i, l in enumerate(labels): by_cls[int(l)].append(i)
        kept = []
        for c in sorted(by_cls):
            pool = by_cls[c]; rng.shuffle(pool)
            kept.extend(pool[:max(1, math.ceil(len(pool)*frac))])
        sub = torch.utils.data.Subset(ds, kept)
        base_t = ds.targets if hasattr(ds, 'targets') else [ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_t[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)
TEST_KW   = dict(batch_size=min(256, len(dataset_test)), num_workers=2, pin_memory=True, shuffle=False)
train_loader = torch.utils.data.DataLoader(dataset_train, **LOADER_KW)
test_loader  = torch.utils.data.DataLoader(dataset_test,  **TEST_KW)
print(f'train_loader: {len(train_loader)} batches  test_loader: {len(test_loader)} batches')

## B-helper. Three-Metric Eval Harness (Index-Based)

Matches the paper's Table 1 / Table 3 protocol exactly.  
All three functions operate on **sample index sets** (not class labels)  
because the 3:7 forget set spans every class.

| Function | Metric |
|----------|--------|
| `eval_output_on_indices` | Output accuracy (model's own classifier head) |
| `eval_probe_on_indices`  | Linear Probe — fresh head trained on full train features |
| `eval_ncc_on_indices`    | NCC — nearest-class-center using per-class mean features |

In [ ]:
# ── Test-set retain/forget split helper ──────────────────────────────
def _test_split_30_70(yte_numpy):
    """Split test indices into retain (70%) / forget (30%) per class, seed=0.
    Identical split used by probe and NCC so comparisons are apples-to-apples."""
    rng = random.Random(0)
    by_cls = collections.defaultdict(list)
    for i, l in enumerate(yte_numpy): by_cls[int(l)].append(i)
    fgt, ret = [], []
    for c in sorted(by_cls):
        pool = list(by_cls[c]); rng.shuffle(pool)
        nf = max(1, round(len(pool) * 0.30))
        fgt.extend(pool[:nf]); ret.extend(pool[nf:])
    return ret, fgt


# ── Metric 1: Output accuracy over arbitrary index set ────────────────
def eval_output_on_indices(model, dataset, indices, device, batch_size=256):
    """Standard forward-pass accuracy over a sample index set."""
    if len(indices) == 0: return float('nan')
    loader = torch.utils.data.DataLoader(
        SubSet(dataset, indices), batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    return correct / max(1, total)


# ── Metric 2: Linear Probe (paper section 3.2) ────────────────────────
@torch.no_grad()
def _extract_features(model, loader, device):
    """Extract penultimate features using the model's extract_features() method."""
    model.eval(); Xs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        # Use extract_features if available (CMF models), else penultimate hook
        if hasattr(model, 'extract_features'):
            f = model.extract_features(x)
        else:
            f = model(x)  # fallback for non-CMF models with FC removed
        Xs.append(f.cpu()); ys.append(y)
    return torch.cat(Xs, 0).float(), torch.cat(ys, 0).long()


def eval_probe_on_indices(model, dataset_train, dataset_test,
                          retain_indices, forget_indices,
                          device, num_classes,
                          probe_epochs=100, probe_lr=0.01, batch_size=256):
    """
    Linear Probe accuracy — paper Section 3.2.
    Train a FRESH linear classifier on frozen encoder features from FULL training
    set (D_r union D_f), then evaluate on test set using _test_split_30_70.
    Returns (probe_retain_acc, probe_forget_acc).
    """
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)

    full_train_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_train_loader, device)

    test_loader_probe = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, test_loader_probe, device)

    head = nn.Linear(Xtr.size(1), num_classes).to(device)
    opt  = optim.SGD(head.parameters(), lr=probe_lr, momentum=0.9, weight_decay=0.0)
    ldr_probe = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(probe_epochs):
        head.train()
        for bx, by in ldr_probe:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()

    head.eval()
    with torch.no_grad():
        pred_te = head(Xte.to(device)).argmax(1).cpu()

    retain_test_indices, forget_test_indices = _test_split_30_70(yte.numpy())

    def _idx_acc(idxs):
        if not idxs: return float('nan')
        return float((pred_te[idxs] == yte[idxs]).float().mean())

    for p in model.parameters(): p.requires_grad_(True)
    return _idx_acc(retain_test_indices), _idx_acc(forget_test_indices)


# ── Metric 3: NCC — Nearest Class Center (paper eq. 5) ───────────────
def eval_ncc_on_indices(model, dataset_train, dataset_test,
                        retain_indices, forget_indices,
                        device, num_classes, batch_size=256):
    """
    NCC accuracy — paper Section 2.1 eq. (5).
    Classify each test sample by nearest class mean (cosine similarity) in
    L2-normalized feature space. Class means from full training set.
    Returns (ncc_retain_acc, ncc_forget_acc).
    """
    model.eval()
    full_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_loader, device)
    Xtr_n = F.normalize(Xtr, dim=1)  # L2-normalize (consistent with CMF feature space)

    class_means = torch.zeros(num_classes, Xtr.size(1))
    for c in range(num_classes):
        mask = (ytr == c)
        if mask.any(): class_means[c] = Xtr_n[mask].mean(0)
    class_means_n = F.normalize(class_means, dim=1)  # [K, D]

    test_loader_ncc = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, test_loader_ncc, device)
    Xte_n = F.normalize(Xte, dim=1)

    with torch.no_grad():
        pred = (Xte_n @ class_means_n.t()).argmax(1)  # [N_test, K] -> argmax

    retain_test_idx, forget_test_idx = _test_split_30_70(yte.numpy())

    def _idx_acc(idxs):
        if not idxs: return float('nan')
        return float((pred[idxs] == yte[idxs]).float().mean())

    return _idx_acc(retain_test_idx), _idx_acc(forget_test_idx)


# ── Combined: all 3 metrics ───────────────────────────────────────────
def eval_three_metrics(model, dataset_train, dataset_test,
                       retain_indices, forget_indices,
                       device, num_classes,
                       run_probe=True, run_ncc=True,
                       probe_epochs=100):
    """
    Compute all 3 paper metrics (Output, Linear Probe, NCC) over index sets.
    Returns a flat dict with all 6 accuracy values.
    """
    # Output
    out_r = eval_output_on_indices(model, dataset_train, retain_indices, device)
    out_f = eval_output_on_indices(model, dataset_train, forget_indices,  device)

    # Linear Probe
    if run_probe and not TEST_MODE:
        pr_r, pr_f = eval_probe_on_indices(
            model, dataset_train, dataset_test,
            retain_indices, forget_indices, device, num_classes,
            probe_epochs=probe_epochs)
    else:
        pr_r = pr_f = float('nan')

    # NCC
    if run_ncc and not TEST_MODE:
        ncc_r, ncc_f = eval_ncc_on_indices(
            model, dataset_train, dataset_test,
            retain_indices, forget_indices, device, num_classes)
    else:
        ncc_r = ncc_f = float('nan')

    return dict(
        output_retain_acc=out_r, output_forget_acc=out_f,
        probe_retain_acc=pr_r,   probe_forget_acc=pr_f,
        ncc_retain_acc=ncc_r,    ncc_forget_acc=ncc_f,
    )


print('Three-metric eval harness defined (Output / Linear Probe / NCC).')


## C. Pre-train Θ_o on Full Training Set

In [ ]:
CKPT_PRETRAIN = f'{CKPT_ROOT}/pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt'
os.makedirs(os.path.dirname(CKPT_PRETRAIN), exist_ok=True)

existing = _find_ckpt(f'pre_train/{DATASET}_{ARCH}_{_MODE_TAG}.pt')
if existing:
    print(f'Checkpoint found: {existing} — skipping training.')
    CKPT_PRETRAIN = existing
else:
    print('Training Θ_o on full training set...')
    args_pt = make_args(
        unlearn_method='pre_train', epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, patience=PRETRAIN_PATIENCE,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        remove_FC=True, CMFClassifier=True,
    )
    model = get_model(args_pt, device)
    optimizer  = optim.SGD(model.parameters(), lr=PRETRAIN_LR,
                           momentum=0.9, weight_decay=5e-4, nesterov=True)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)
    best_acc, train_log = 0.0, []

    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        model.train()
        total_loss = total_n = 0
        # Recompute CMF weights once per epoch from full train set
        if hasattr(model, 'recompute_cmf'):
            model.eval(); model.recompute_cmf(train_loader, device=device); model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0); total_n += x.size(0)
        scheduler.step()
        if epoch % max(1, PRETRAIN_EPOCHS // 10) == 0 or epoch == PRETRAIN_EPOCHS:
            if hasattr(model, 'recompute_cmf'):
                model.eval(); model.recompute_cmf(train_loader, device=device)
            ra, _, _ = test(model, device, test_loader, [],
                            CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')
            print(f'  Epoch {epoch:3d}  loss={total_loss/max(1,total_n):.4f}  test_acc={ra:.4f}')
            train_log.append({'epoch': epoch, 'loss': total_loss/max(1,total_n), 'acc': ra})
            if ra > best_acc:
                best_acc = ra
                torch.save(model.state_dict(), CKPT_PRETRAIN)
                print(f'  → saved best (acc={best_acc:.4f})')
            model.train()

    torch.save(model.state_dict(), CKPT_PRETRAIN)
    with open(f'{CKPT_ROOT}/pre_train/train_log.json', 'w') as f:
        json.dump(train_log, f, indent=2)
    print(f'Pre-train complete → {CKPT_PRETRAIN}')

# Load and evaluate Θ_o
args_pt = make_args(unlearn_method='pre_train', num_classes=NUM_CLASSES,
                    class_label_names=CLASS_LABEL_NAMES, remove_FC=True, CMFClassifier=True)
orig_model = get_model(args_pt, device)
orig_model.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
orig_model.eval()
if hasattr(orig_model, 'recompute_cmf'):
    orig_model.recompute_cmf(train_loader, device=device)
print('\n── Θ_o evaluation ──')
test(orig_model, device, test_loader, [], CLASS_LABEL_NAMES, NUM_CLASSES, set_name='Test')

## D. Build Stratified 3:7 Cross-Class Forget Splits

`build_stratified_random_split` samples exactly 30% of each class's training samples
into the forget set and 70% into retain. Saved as JSON for full reproducibility.
All downstream notebooks load these files — never regenerate.

In [ ]:
def build_stratified_random_split(dataset, forget_fraction=0.30, seed=0):
    """
    Stratified random cross-class forget split.
    Exactly `forget_fraction` of each class's samples -> forget set.
    The rest -> retain set. Forget set is mixed across all classes.
    Returns: forget_indices (list), retain_indices (list), per_class_log (dict)
    """
    rng = random.Random(seed)
    if hasattr(dataset, 'targets'):
        all_targets = dataset.targets
    elif hasattr(dataset, 'indices'):
        all_targets = [dataset.dataset.targets[i] for i in dataset.indices]
    else:
        all_targets = [dataset[i][1] for i in range(len(dataset))]

    by_class = collections.defaultdict(list)
    for idx, lbl in enumerate(all_targets): by_class[int(lbl)].append(idx)

    forget_indices, retain_indices, per_class_log = [], [], {}
    for cls in sorted(by_class):
        pool = list(by_class[cls]); rng.shuffle(pool)
        nf = round(len(pool) * forget_fraction)
        nf = max(1, min(nf, len(pool) - 1))
        forget_indices.extend(pool[:nf])
        retain_indices.extend(pool[nf:])
        per_class_log[cls] = {
            'total': len(pool), 'forget': nf,
            'retain': len(pool) - nf,
            'forget_pct': f'{100*nf/len(pool):.1f}%',
        }
    return forget_indices, retain_indices, per_class_log


for seed in SPLIT_SEEDS:
    forget_idx, retain_idx, per_class_log = build_stratified_random_split(
        dataset_train, forget_fraction=FORGET_FRACTION, seed=seed)

    out_path = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    with open(out_path, 'w') as f:
        json.dump({
            'seed': seed, 'forget_fraction': FORGET_FRACTION,
            'dataset': DATASET,
            'total_train': len(dataset_train),
            'n_forget': len(forget_idx), 'n_retain': len(retain_idx),
            'forget_indices': forget_idx, 'retain_indices': retain_idx,
            'per_class': per_class_log,
        }, f)

    print(f'\n── Seed {seed} ─────────────────────────────────────────────')
    print(f'  total={len(dataset_train)}  forget={len(forget_idx)}  '
          f'retain={len(retain_idx)}  '
          f'overall_forget%={100*len(forget_idx)/len(dataset_train):.1f}%')
    hdr = f'{"class":>5} {"total":>7} {"forget":>7} {"retain":>7} {"pct":>6}'
    print('  ' + hdr + '\n  ' + '-'*len(hdr))
    for cls, info in sorted(per_class_log.items()):
        print(f'  {cls:>5} {info["total"]:>7} {info["forget"]:>7} '
              f'{info["retain"]:>7} {info["forget_pct"]:>6}')
    print(f'  → {out_path}')

print(f'\nAll {len(SPLIT_SEEDS)} splits saved to {SPLIT_DIR}/')

## E. Save Config & Evaluate Θ_o on All Splits

In [ ]:
CONFIG = {
    'TEST_MODE': TEST_MODE, 'TEST_FRACTION': TEST_FRACTION, '_MODE_TAG': _MODE_TAG,
    'DATASET': DATASET, 'ARCH': ARCH, 'IS_VIT': IS_VIT,
    'NUM_CLASSES': NUM_CLASSES, 'CLASS_LABEL_NAMES': CLASS_LABEL_NAMES,
    'TOTAL': _TOTAL[DATASET], 'PER_CLASS': _PER_CLASS[DATASET],
    'SPLIT_SEEDS': SPLIT_SEEDS, 'FORGET_FRACTION': FORGET_FRACTION, 'SPLIT_DIR': SPLIT_DIR,
    'PRETRAIN_LR': PRETRAIN_LR, 'PRETRAIN_EPOCHS': PRETRAIN_EPOCHS,
    'PRETRAIN_BS': PRETRAIN_BS, 'PRETRAIN_PATIENCE': PRETRAIN_PATIENCE,
    'CKPT_ROOT': CKPT_ROOT, 'CKPT_PRETRAIN': CKPT_PRETRAIN,
}
config_path = f'{CKPT_ROOT}/cmf_benchmark_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)
print(f'Config saved: {config_path}')

In [ ]:
# Evaluate Θ_o on all 3 seeds using all 3 metrics
# This produces the 'Original' row of the paper's Table 1
import pandas as pd
orig_eval_rows = []
for seed in SPLIT_SEEDS:
    with open(f'{SPLIT_DIR}/forget_indices_seed{seed}.json') as f:
        sp = json.load(f)
    retain_idx = sp['retain_indices']
    forget_idx = sp['forget_indices']

    # Reload Θ_o fresh
    m = get_model(args_pt, device)
    m.load_state_dict(torch.load(CKPT_PRETRAIN, map_location=device))
    if hasattr(m, 'recompute_cmf'):
        m.eval(); m.recompute_cmf(train_loader, device=device)

    metrics = eval_three_metrics(
        m, dataset_train, dataset_test,
        retain_idx, forget_idx, device, NUM_CLASSES,
        run_probe=True, run_ncc=True, probe_epochs=50 if TEST_MODE else 100)

    row = {'model': 'Original', 'seed': seed, **metrics}
    orig_eval_rows.append(row)
    print(f'Seed {seed}: output R={metrics["output_retain_acc"]:.4f} '
          f'F={metrics["output_forget_acc"]:.4f} | '
          f'probe R={metrics["probe_retain_acc"]:.4f} '
          f'F={metrics["probe_forget_acc"]:.4f} | '
          f'NCC R={metrics["ncc_retain_acc"]:.4f} '
          f'F={metrics["ncc_forget_acc"]:.4f}')

orig_df = pd.DataFrame(orig_eval_rows)
csv_path = f'{CKPT_ROOT}/results_original_{DATASET}_{ARCH}.csv'
orig_df.to_csv(csv_path, index=False)

# Summary (mean ± std over seeds)
metric_cols = ['output_retain_acc','output_forget_acc',
               'probe_retain_acc','probe_forget_acc',
               'ncc_retain_acc','ncc_forget_acc']
print('\n=== Θ_o Evaluation — mean ± std over seeds (matches paper Table 1 "Original" row) ===')
for col in metric_cols:
    vals = orig_df[col].dropna()
    if len(vals): print(f'  {col:30s}: {vals.mean():.4f} ± {vals.std():.4f}')

print(f'\nResults saved: {csv_path}')

## Summary

**Outputs ready for downstream notebooks:**

| File | Description |
|------|-------------|
| `checkpoints/cmf_benchmark/pre_train/<tag>.pt` | Θ_o — original model on full training set |
| `checkpoints/cmf_benchmark/splits/forget_indices_seed{0,1,2}.json` | 3-seed 30%/70% stratified cross-class splits |
| `checkpoints/cmf_benchmark/cmf_benchmark_config.json` | All config for downstream notebooks |
| `checkpoints/cmf_benchmark/results_original_*.csv` | Θ_o 3-metric eval (Output/Probe/NCC) on all seeds |

**Next:** Publish `checkpoints/cmf_benchmark/` as a Kaggle dataset, attach to NB2/NB3/NB4.